In [6]:
def insert(name:str,age:int): # this just hints the type doesnt raises error
    print(name)
    print(age)

In [7]:
def insert(name:str,age:int):
    # we could do it manually but its just not scalable and right at production lvl
    if type(name)==str and type(age)==int:

        print(name)
        print(age)
    else:
        raise TypeError('incorrect data type')
insert('hi','why')

TypeError: incorrect data type

# MAIN 2 PROBS
- type validation
- data validation

# Now come pYdnatic


In [8]:
import pydantic
print(pydantic.__version__)

2.13.4


In [9]:
from pydantic import BaseModel

class Patient(BaseModel):
    name:str  # now here we r not using constructor ,that is directly these attributes are not formed
    age:int    # these are created when values assinged where as constructor creates those on the object creation moment
    # though still at back , Base Model does do us __init__ to establish these attributes
patient=Patient(name='ABC',age=23)
#or
data={'name':'bin','age':'23'} # even this works that is if type conversion is possible then its done
patient=Patient(**data)

In [10]:
def insert(patient:Patient):
    print(patient.name)
    print(patient.age)
insert(patient)

bin
23


Lets Level Up

In [11]:
from typing import (List,Dict,Optional,
                    Annotated)
from pydantic import (
EmailStr,
AnyUrl,
Field ## for data validation and setting meta data 

)
# combining annotated and field we get to do various meta data additions

class Patient(BaseModel):

    # name:str
    name:Annotated[str,Field(max_length=100,title='Patients Name',description='Enter the patients Name',examples=['rohit'])]
    age:int =Field(gt=0) # age validation
    gender:str


    weight_kg:Annotated[float,Field(strict=True)] # this strict flag stops the auto type conversion we witnessed 

    height_cm:int
    blood_group:str
    city:str
    state:Optional[str]=None
    phone:str
    email:EmailStr
    occupation:Optional[str]=None

    # married:bool=False  # assigning default values
    #or
    married:Annotated[bool,Field(default=None)]

    allergies:List[str]
    medical_conditions:Optional[List[str]]=None
    emergency_contact:Optional[Dict[str,str]]=None # as we want to validate the inner data as well so we Typing Module
    address:Optional[str]=None
    # address:str=None ---> this cant be done 

    linkedin_url:Optional[AnyUrl]=None



In [12]:
p1={
    "name": "Aarav Sharma",
    "age": 29,
    "gender": "Male",
    "height_cm": 175,
    "weight_kg": 72,
    "blood_group": "B+",
    "city": "Delhi",
    "state": "Delhi",
    "phone": "+91-9876543201",
    "email": "aarav.sharma@example.com",
    "occupation": "Software Engineer",
    "married": False ,
    "allergies": ["Penicillin"],
    "medical_conditions": [],
    "emergency_contact": {
      "name": "Rohit Sharma",
      "relation": "Brother",
      "phone": "+91-9876543210"
    }
  }
patient=Patient(**p1)

Field Validator

- now suppose we req to validate custom data eg email being from particular org 
- or we req to do some transformations on data (eg capilisation)

In [ ]:
from pydantic import field_validator

class Patient(BaseModel):

    name:str
    age:int
    email:EmailStr

    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        domain=value.split('@')[-1]
        if domain not in ['hdfc.com','axis.com','icici.com']:
            raise ValueError('Domain Not in DB')
        return value
    
    @field_validator('name') # using for data transformation
    @classmethod
    def caps(cls,value):
        return value.upper()
    
    # another specific detail is that field validator has a mode either after or before ,where the value we get
    #in function in before is , prior to type validation
    # by default its after

    @field_validator('age',mode='after') # with before '23' doesnt works
    @classmethod
    def age_check(cls,value):
        if value>0 and value<100:
            return value
        else:
            raise ValueError('Incorrect value of age provided')




data={'name':'bin','age':'23','email':'hey@axis.com'}
p=Patient(**data)
print(p.name)

BIN


Model Validator

now if we req to do some data validation based on multiple feilds we use it

In [24]:
from pydantic import model_validator

class Patient(BaseModel):

    name:str
    age:int
    email:EmailStr
    weight:float

    @model_validator(mode='after')
    @classmethod
    def check(cls,model):
        if model.age>60 and model.weight>100:
            raise ValueError('Person is old  and obese')
        else:
            return model
data={'name':'bin','age':34,'email':'hey@axis.com','weight':110}
p=Patient(**data)
print(p.name)

bin


C:\Users\nishc\AppData\Local\Temp\ipykernel_41588\4244879846.py:10: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


Computed Fields

- its simply just the ops on data obtained to create new fields 
(may be take ref with how we work with colunms and fields in a DB),eg BMI 

In [ ]:
# why cant we just use simple function here?
# why @class method? nd why not here
from pydantic import computed_field

# the main distinction is that computed field in serialisation would add the defined method's output
# where as where as simple class method it wont

class Patient(BaseModel):

    name:str
    age:int
    email:EmailStr
    weight:float
    height:float

    @computed_field
    def bmi(self) -> float:  # return type is req here
        bmi=self.weight/self.height**2
        return bmi
    
data={'name':'bin','age':34,'email':'hey@axis.com','weight':70,'height':1.7}
p=Patient(**data)
print(p.name)
print(p.bmi)

bin
24.221453287197235


Nested Models

when we use one model in other model as a field

In [ ]:
# better and re usable code segments

class Address(BaseModel):
    city:str
    state:str
    pin:str

class Patient(BaseModel):

    name:str
    age:int
    weight:float
    address:Address

address={'city':'a','state':'jio','pin':'12344'}

data={'name':'bin','age':34,'weight':70,'address':Address(**address)}
p=Patient(**data)
print(p.name)
print(p.address)
print(p.address.pin)

bin
city='a' state='jio' pin='12344'
12344


Serialisation

- this is just the segment where the generated data in objects can be extracted and exported 
as dic or json

In [41]:
temp=p.model_dump()
print(temp)
type(temp)

{'name': 'bin', 'age': 34, 'weight': 70.0, 'address': {'city': 'a', 'state': 'jio', 'pin': '12344'}}


dict

In [ ]:
temp=p.model_dump_json()
print(temp)
print(type(temp)) # as this str then can be loaded using json.load()

{"name":"bin","age":34,"weight":70.0,"address":{"city":"a","state":"jio","pin":"12344"}}
<class 'str'>


In [44]:
temp=p.model_dump(include=['name'])
print(temp)

{'name': 'bin'}


In [45]:
temp=p.model_dump(exclude=['weight'])
print(temp)

{'name': 'bin', 'age': 34, 'address': {'city': 'a', 'state': 'jio', 'pin': '12344'}}


In [47]:
temp=p.model_dump(exclude={'address':'state'})
print(temp)

{'name': 'bin', 'age': 34, 'weight': 70.0, 'address': {'city': 'a', 'pin': '12344'}}


In [ ]:
temp=p.model_dump(exclude_unset=True) # excludes the unset object values
print(temp)

{'name': 'bin', 'age': 34, 'weight': 70.0, 'address': {'city': 'a', 'state': 'jio', 'pin': '12344'}}
